# NFL Big Data Bowl 2027: Position-Selective Kinematic Pipeline
This interactive notebook provides an end-to-end framework linking **10 Hz optical tracking sensors** at the Scouting Combine (`combine_tracking.csv`) to **regular-season NFL performance** (`player_play.csv`) and **career outcomes** (`player_career_successes.csv`).

### Pipeline Architecture
1. **Position Selection & Parameterization**: Choose any position group (e.g., `WR`, `CB`, `DE`, `OLB`, `T`, `RB`).
2. **Data Ingestion**: Loads or generates statistically calibrated synthetic data matching the official competition schema.
3. **Sensor Feature Engineering**: Extracts peak acceleration, top speed, burst velocity, and tracking deltas.
4. **Position-Adaptive Target Mapping**: Links sensor features to position-relevant in-game micro targets.
5. **Predictive Benchmarking**: Compares Traditional Combine metrics vs Kinematic Sensor metrics using Cross-Validated \(R^2\).
6. **Scouting Dashboard**: Visualizes time-series trajectories, feature correlations, model benchmarks, and prospect rankings.

In [ ]:
# CELL 1: Pipeline Configuration & Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 6)

====================================================================
# 🎯 POSITION SELECTION CONFIGURATION
# Options: 'WR', 'CB', 'DE', 'OLB', 'RB', 'TE', 'T', 'G', 'C', 'DT'
====================================================================
SELECTED_POSITION = 'WR'

POSITION_CONFIGS = {
    'WR': {'category': 'Offensive Skill', 'drill': 'FORTY_YARD_DASH', 'micro_target': 'separation_at_pass_forward', 'macro_target': 'career_offensive_snaps'},
    'CB': {'category': 'Defensive Skill', 'drill': 'FORTY_YARD_DASH', 'micro_target': 'cushion', 'macro_target': 'career_defensive_snaps'},
    'DE': {'category': 'Pass Rush / Defensive Line', 'drill': 'FORTY_YARD_DASH', 'micro_target': 'player_get_off', 'macro_target': 'career_defensive_snaps'},
    'OLB': {'category': 'Pass Rush / Linebacker', 'drill': 'FORTY_YARD_DASH', 'micro_target': 'time_to_pressure', 'macro_target': 'career_defensive_snaps'},
    'RB': {'category': 'Offensive Skill', 'drill': 'FORTY_YARD_DASH', 'micro_target': 'yards_after_catch', 'macro_target': 'career_offensive_snaps'},
    'T': {'category': 'Offensive Line', 'drill': 'FORTY_YARD_DASH', 'micro_target': 'time_to_pressure_allowed', 'macro_target': 'career_offensive_snaps'},
}

pos_cfg = POSITION_CONFIGS.get(SELECTED_POSITION, POSITION_CONFIGS['WR'])
print(f"🎯 PIPELINE CONFIGURED FOR POSITION: {SELECTED_POSITION}")
print(f"   - Category: {pos_cfg['category']}")
print(f"   - Primary Combine Drill: {pos_cfg['drill']}")
print(f"   - Target Micro Metric (In-Game): {pos_cfg['micro_target']}")
print(f"   - Target Macro Metric (Career): {pos_cfg['macro_target']}")

In [ ]:
# CELL 2: Data Ingestion & Synthetic Schema Generator
def load_or_generate_data(position='WR', num_players=60):
    np.random.seed(42)
    nfl_ids = np.arange(55000, 55000 + num_players)
    
    # 1. Players
    players_df = pd.DataFrame({
        'nfl_id': nfl_ids,
        'display_name': [f'Prospect_{i}' for i in range(num_players)],
        'draft_year': np.random.choice([2023, 2024, 2025], size=num_players),
        'nfl_position': position,
        'college_conference': np.random.choice(['SEC', 'Big Ten', 'ACC', 'Big 12', 'Pac-12'], size=num_players),
        'draft_round': np.random.choice([1, 2, 3, 4, 5, 6, 7], size=num_players),
        'draft_overall_pick': np.random.randint(1, 250, size=num_players)
    })
    
    # 2. Combine Results
    combine_results_df = pd.DataFrame({
        'nfl_id': nfl_ids,
        'draft_year': players_df['draft_year'],
        'combine_height': np.random.normal(74, 2.5, num_players),
        'combine_weight': np.random.normal(210, 20, num_players),
        'ten_yd_split': np.random.normal(1.55, 0.05, num_players),
        'forty': np.random.normal(4.48, 0.12, num_players),
        'vertical': np.random.normal(35, 3, num_players),
        'broad_jump': np.random.normal(120, 6, num_players),
        'three_cone': np.random.normal(7.0, 0.25, num_players),
        'short_shuttle': np.random.normal(4.2, 0.15, num_players),
        'ngs_athleticism_score': np.random.randint(50, 99, num_players),
        'ngs_college_production_score': np.random.uniform(50, 95, num_players),
        'ngs_final_score': np.random.uniform(55, 95, num_players)
    })
    
    # 3. Combine Tracking Frames (10 Hz for 40-yd dash ~4.5 seconds = 50 frames per player)
    tracking_rows = []
    for pid in nfl_ids:
        forty_time = combine_results_df.loc[combine_results_df['nfl_id'] == pid, 'forty'].values[0]
        n_frames = int(forty_time * 10)
        t_steps = np.linspace(0, forty_time, n_frames)
        x_coords = 40.0 * (t_steps / forty_time)**1.3  # non-linear acceleration
        speeds = np.gradient(x_coords, t_steps[1] - t_steps[0])
        accels = np.gradient(speeds, t_steps[1] - t_steps[0])
        for idx, t in enumerate(t_steps):
            tracking_rows.append({
                'nfl_id': pid,
                'event_id': f'combine_2025_{pid}',
                'drill_type': 'FORTY_YARD_DASH',
                'time_step': t,
                'x': x_coords[idx],
                'y': 26.66,
                's': max(0.0, speeds[idx]),
                'a': accels[idx],
                'dis': speeds[idx] * 0.1,
                'dir': 90.0
            })
    combine_tracking_df = pd.DataFrame(tracking_rows)
    
    # 4. In-Game Plays (player_play)
    play_rows = []
    for pid in nfl_ids:
        n_plays = np.random.randint(15, 50)
        for p in range(n_plays):
            play_rows.append({
                'nfl_id': pid,
                'game_id': 2024000000 + np.random.randint(100, 900),
                'play_id': p + 1,
                'separation_at_pass_forward': np.random.normal(2.8, 0.8),
                'cushion': np.random.normal(6.0, 1.5),
                'player_get_off': np.random.normal(0.75, 0.10),
                'time_to_pressure': np.random.normal(2.5, 0.4),
                'time_to_pressure_allowed': np.random.normal(2.6, 0.5),
                'yards_after_catch': np.random.normal(4.5, 2.0)
            })
    player_play_df = pd.DataFrame(play_rows)
    
    # 5. Career Outcomes
    career_df = pd.DataFrame({
        'nfl_id': nfl_ids,
        'career_offensive_snaps': np.random.randint(50, 1200, size=num_players),
        'career_defensive_snaps': np.random.randint(50, 1200, size=num_players),
        'career_games_active': np.random.randint(5, 34, size=num_players),
        'career_games_started': np.random.randint(0, 30, size=num_players)
    })
    
    return players_df, combine_results_df, combine_tracking_df, player_play_df, career_df

players_df, combine_results_df, combine_tracking_df, player_play_df, career_df = load_or_generate_data(SELECTED_POSITION)
print(f"✓ Data ingestion complete for {SELECTED_POSITION}:\n"
      f"   - Master Players: {players_df.shape}\n"
      f"   - Combine Results: {combine_results_df.shape}\n"
      f"   - Combine Tracking Frames: {combine_tracking_df.shape}\n"
      f"   - In-Game Play Records: {player_play_df.shape}\n"
      f"   - Career Outcomes Records: {career_df.shape}")

In [ ]:
# CELL 3: Sensor Kinematic Feature Extraction (Combine Tracking)
def extract_combine_kinematic_features(tracking_df, combine_res_df):
    features = []
    grouped = tracking_df.groupby('nfl_id')
    
    for nfl_id, group in grouped:
        group = group.sort_values('time_step')
        
        # Peak acceleration & top speed
        peak_accel = group['a'].max()
        top_speed = group['s'].max()
        
        # Time to top speed
        top_speed_time = group.loc[group['s'] == top_speed, 'time_step'].values[0] if len(group) > 0 else np.nan
        
        # Sensor estimated 10-yard split (time where x >= 10)
        yd_10 = group[group['x'] >= 10.0]
        sensor_10yd_split = yd_10['time_step'].iloc[0] if len(yd_10) > 0 else np.nan
        
        # Velocity maintenance (speed at end / top speed)
        end_speed = group['s'].iloc[-1] if len(group) > 0 else np.nan
        speed_maintenance_ratio = end_speed / top_speed if top_speed > 0 else np.nan
        
        features.append({
            'nfl_id': nfl_id,
            'kinematic_peak_accel': peak_accel,
            'kinematic_top_speed': top_speed,
            'kinematic_time_to_top_speed': top_speed_time,
            'sensor_10yd_split': sensor_10yd_split,
            'kinematic_speed_maintenance': speed_maintenance_ratio
        })
        
    feat_df = pd.DataFrame(features)
    
    # Merge with combine_results to calculate tracking delta
    feat_df = feat_df.merge(combine_res_df[['nfl_id', 'ten_yd_split', 'forty']], on='nfl_id', how='left')
    feat_df['split_tracking_delta'] = feat_df['sensor_10yd_split'] - feat_df['ten_yd_split']
    
    return feat_df

kinematic_features_df = extract_combine_kinematic_features(combine_tracking_df, combine_results_df)
print(f"✓ Kinematic Feature Extraction Complete. Shape: {kinematic_features_df.shape}")
display(kinematic_features_df.head())

In [ ]:
# CELL 4: Visualizing 10 Hz Kinematic Profiles (Intermediate Output 1)
sample_players = kinematic_features_df['nfl_id'].unique()[:4]
plt.figure(figsize=(14, 5))

for pid in sample_players:
    p_track = combine_tracking_df[combine_tracking_df['nfl_id'] == pid].sort_values('time_step')
    p_name = players_df.loc[players_df['nfl_id'] == pid, 'display_name'].values[0]
    plt.plot(p_track['time_step'], p_track['s'], label=f'{p_name} (Speed yd/s)', linewidth=2)

plt.title(f'10 Hz Sensor Velocity Curves during 40-Yard Dash ({SELECTED_POSITION})', fontsize=14, fontweight='bold')
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Speed (yards/sec)', fontsize=12)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 5: In-Game Performance Target Aggregation & Dataset Merging
micro_target_col = pos_cfg['micro_target']
macro_target_col = pos_cfg['macro_target']

# Aggregate in-game player metrics per player
in_game_agg = player_play_df.groupby('nfl_id').agg(
    micro_target_mean=(micro_target_col, 'mean'),
    micro_target_std=(micro_target_col, 'std'),
    snaps_analyzed=(micro_target_col, 'count')
).reset_index()

# Rename aggregated column dynamically
in_game_agg = in_game_agg.rename(columns={'micro_target_mean': f'micro_{micro_target_col}_mean'})

# Construct Master Dataset
master_df = players_df.merge(combine_results_df, on=['nfl_id', 'draft_year'], how='left') \
                      .merge(kinematic_features_df, on='nfl_id', how='left') \
                      .merge(in_game_agg, on='nfl_id', how='left') \
                      .merge(career_df[['nfl_id', macro_target_col]], on='nfl_id', how='left')

master_df = master_df.dropna(subset=[f'micro_{micro_target_col}_mean'])
print(f"✓ Master Merged Dataset Built for {SELECTED_POSITION}. Shape: {master_df.shape}")
print(f"   - Micro Target: micro_{micro_target_col}_mean")
print(f"   - Macro Target: {macro_target_col}")
display(master_df.head(3))

In [ ]:
# CELL 6: Feature Correlation Matrix (Intermediate Output 2)
feature_cols = [
    'ten_yd_split', 'forty', 'vertical', 'broad_jump',
    'kinematic_peak_accel', 'kinematic_top_speed', 'kinematic_time_to_top_speed',
    'split_tracking_delta', f'micro_{micro_target_col}_mean'
]

corr_matrix = master_df[feature_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title(f'Feature Correlation Heatmap: Combine Metrics vs In-Game Target ({SELECTED_POSITION})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Predictive Benchmarking — Traditional vs Kinematic Models
target_y = master_df[f'micro_{micro_target_col}_mean'].values

trad_features = ['ten_yd_split', 'forty', 'vertical', 'broad_jump', 'combine_height', 'combine_weight']
kin_features = ['kinematic_peak_accel', 'kinematic_top_speed', 'kinematic_time_to_top_speed', 'split_tracking_delta', 'kinematic_speed_maintenance']

X_trad = master_df[trad_features].fillna(master_df[trad_features].median()).values
X_kin = master_df[kin_features].fillna(master_df[kin_features].median()).values

def evaluate_model(X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    r2_scores, rmse_scores = [], []
    preds = np.zeros_like(y)
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=4)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        preds[val_idx] = y_pred
        
        r2_scores.append(r2_score(y_val, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))
        
    return np.mean(r2_scores), np.mean(rmse_scores), preds

trad_r2, trad_rmse, trad_preds = evaluate_model(X_trad, target_y)
kin_r2, kin_rmse, kin_preds = evaluate_model(X_kin, target_y)

print(f"=====================================================")
print(f"📊 PREDICTIVE MODEL PERFORMANCE BENCHMARK ({SELECTED_POSITION})")
print(f"   1. Traditional Combine Model   -> CV R²: {trad_r2:.3f} | CV RMSE: {trad_rmse:.3f}")
print(f"   2. Kinematic Sensor Model     -> CV R²: {kin_r2:.3f} | CV RMSE: {kin_rmse:.3f}")
print(f"   -> Delta R² Improvement: {kin_r2 - trad_r2:+.3f}")
print(f"=====================================================")

In [ ]:
# CELL 8: Visualizing Predictive Benchmarks (Intermediate Output 3)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart Comparison
models = ['Traditional Combine', 'Kinematic Sensor']
r2_vals = [trad_r2, kin_r2]
colors = ['#1f77b4', '#ff7f0e']

ax[0].bar(models, r2_vals, color=colors, width=0.5)
ax[0].set_title('Cross-Validated R² Metric Comparison', fontsize=12, fontweight='bold')
ax[0].set_ylabel('CV R² Score')
ax[0].axhline(0, color='gray', linestyle='--')

# Actual vs Predicted Scatter Plot
ax[1].scatter(target_y, kin_preds, color='#2ca02c', alpha=0.7, edgecolors='k', label='Kinematic Model')
ax[1].plot([target_y.min(), target_y.max()], [target_y.min(), target_y.max()], 'r--', label='Ideal Fit')
ax[1].set_title(f'Actual vs Predicted In-Game Target ({micro_target_col})', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Actual In-Game Metric')
ax[1].set_ylabel('Model Predicted Metric')
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# CELL 9: Feature Importance Analysis
X_all = master_df[trad_features + kin_features].fillna(0)
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_all, target_y)

importances = pd.Series(model_rf.feature_importances_, index=trad_features + kin_features).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#9467bd')
plt.title(f'Random Forest Feature Importance for In-Game Metric ({SELECTED_POSITION})', fontsize=13, fontweight='bold')
plt.xlabel('Relative Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 10: Prospect Kinematic Scouting Leaderboard
master_df['kinematic_composite_score'] = (
    master_df['kinematic_peak_accel'].rank(pct=True) * 0.4 +
    master_df['kinematic_top_speed'].rank(pct=True) * 0.4 +
    (1 - master_df['split_tracking_delta'].rank(pct=True)) * 0.2
) * 100.0

scouting_board = master_df[[
    'nfl_id', 'display_name', 'draft_year', 'draft_overall_pick',
    'forty', 'kinematic_peak_accel', 'kinematic_top_speed', 'kinematic_composite_score',
    f'micro_{micro_target_col}_mean', macro_target_col
]].sort_values('kinematic_composite_score', ascending=False)

print("=====================================================")
print(f"🏆 TOP 10 PROSPECT KINEMATIC SCOUTING RANKINGS ({SELECTED_POSITION})")
print("=====================================================")
display(scouting_board.head(10))